# 📋 STR01 · 미션 08 | 전면전 발발 모의 전 부대 보급품 동시 급감 스트레스 시뮬레이션

**학생용 Student · 직접 실습**  
CMS 편성: 3시간 · 교육용 가상 데이터 사용

---

## 📖 1. 상황과 이번 미션의 문제

가상의 20개 조직과 5종 물품의 기록을 이용해 **전면전 발발 모의 전 부대 보급품 동시 급감 스트레스 시뮬레이션** 과제를 수행합니다.

실제 조직의 기록이 아닌 실습자료이며, 코드를 실행한 결과도 실제 현장 상태를 뜻하지 않습니다.

**이번 미션에서 해결할 문제:** 소모를 1.5배, 2배, 3배로 키우고 보급량을 절반으로 줄이는 충격을 12일간 가정해 재고를 다시 계산한다.

모델만 쓸 때와 단기 소모 기반 안전장치를 더했을 때의 경보 발령 정확도와 미탐률을 비교하고, 상황판 1회 갱신 시간을 50회 반복 측정한다.

이는 교육용 가상 모델이며 실제 작전 계획과 무관하다.

**만들 결과물:** `src/dashboard/str01_app.py`, `reports/str01_stress_report.md`, `outputs/M08/res.csv`, `forms/STR01_M08_forms.md`

## 📌 2. 처음 시작하는 방법

1. 아래 **환경 준비 셀**을 실행합니다. Colab에서는 메뉴의 **런타임 → 모두 실행** 대신 각 셀의 결과를 먼저 확인하면서 진행해도 됩니다.
2. 핵심 함수의 `TODO`를 AI와 함께 구현합니다. 처음에는 함수의 입력·출력과 왜 필요한지 질문하세요.
3. 첫 실행 결과를 확인하고 실패하면 오류 메시지를 AI에게 설명해 달라고 요청합니다.
4. AI가 제안한 코드 중 최소 3가지를 원자료와 실행 결과로 대조하고 수정 이유를 기록합니다.
5. 활동별 작성 칸을 채운 뒤 **실제 AI 대화 원문**을 보존하고 마지막 저장 셀로 ZIP을 만듭니다.

> **보안과 평가:** 실제 조직·위치·재고·작전 자료, 개인정보를 외부 AI에 입력하지 않습니다.

- 모델 성능은 **실제 출력 로그**로 판단하며, 예시값을 실측으로 적지 않습니다.
- 실행 성공이나 ZIP 저장은 학습 PASS·실제 승인과 다릅니다.
- 폐쇄망에서는 허용된 Python/Jupyter 환경에서 실행하세요.
- 인터넷이 필요한 `pip` 설치는 폐쇄망에서 실패할 수 있으므로 사전 패키지 준비가 필요합니다.


In [ ]:
# @title ① 실습자료 자동 준비 및 환경 설정 (ZIP 업로드 불필요) (실행 버튼을 누르세요) { display-mode: "form" }
# 이 셀은 코드 숨김 상태로 배포됩니다. 소스코드를 열지 않아도 실행할 수 있습니다.
from pathlib import Path
import os, sys, json, re, math, time, warnings, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

# 업로드 파일명은 자유이며, .zip 확장자와 유효한 ZIP 내용만 확인합니다.
current_dir = Path.cwd().resolve()
notebook_directory = globals().get('notebook_directory', current_dir)
workspace_name = 'STR01_실습프로젝트_작업공간'
WORKSPACE = next(
    (folder for folder in (*reversed(current_dir.parents), current_dir) if folder.name == workspace_name),
    current_dir / workspace_name,
)
STR01_PROJECT_READY = False

def _prepare_project_zip(raw_bytes):
    global STR01_PROJECT_READY, STR01_PROJECT_DIR
    import io
    if not zipfile.is_zipfile(io.BytesIO(raw_bytes)):
        raise ValueError('유효한 ZIP 파일이 아닙니다. 압축파일 내용을 확인하세요.')
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw_bytes)) as archive:
        for member in archive.infolist():
            name = member.filename.replace('\\', '/')
            target = (WORKSPACE / name).resolve()
            if not target.is_relative_to(WORKSPACE):
                raise ValueError('안전하지 않은 압축 경로가 발견되어 중단했습니다: ' + name)
            # 심볼릭 링크는 작업공간 외부 접근 가능성을 고려해 거부합니다.
            if (member.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('심볼릭 링크가 포함된 ZIP은 허용하지 않습니다: ' + name)
        for member in archive.infolist():
            target = (WORKSPACE / member.filename.replace('\\', '/')).resolve()
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            if target.is_file():
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, target.open('xb') as destination:
                import shutil
                shutil.copyfileobj(source, destination)
    os.chdir(WORKSPACE)
    STR01_PROJECT_DIR = WORKSPACE
    STR01_PROJECT_READY = True
    print('✅ 실습자료 준비 완료 (기존 파일은 보존):', WORKSPACE)
    print('✅ 이제 다음 실습 코드셀을 실행할 수 있습니다.')

# 실습자료가 노트북 안에 내장되어 있어 ZIP 업로드 없이 자동으로 준비됩니다.
# 내장 파일 2개: STR01_M08_forms.md, STR01_M08_lesson.md
import base64 as _embedded_b64
_EMBEDDED_ZIP_B64 = (
    'UEsDBBQAAAAIAAAAIVyn3TrMjQgAAFsUAAASAAAAU1RSMDFfTTA4X2Zvcm1zLm1k3VhNbxtHEr3zVzRg5CaJsh0DG12CHAMkOWxyCxaCNlY+sPFKEO0AC+hA'
    'WUODIWmYsjnSSB4SQ4eySC+zHpIjkULkHPRTnNt0z3/YV9Xdw5FEOTlkLwsRItnTM10fr1694g3x+Rd/XbwpPl38i1CtHeUMhNrdUdWmeFtsCBU4shfhv5Ch'
    'j5eQr7qq6QleOSnKWlHIURRPfkyeloV8sq+qvsC3OKwLVekklYkMyvggsC7/06cvzUiVvFxOPa/L0UQkT7C3uCS+VE6QbDf/JtRJLx72eP9DT77sK69zPpZt'
    'XzZ4MQ4dwYbhZNoziuQzGHXh+i0VlQXOkQ+jOREP32ATWx+cwmw32XflE0d8cOc92qO2t2hnL5zDN0897MfDEJuEOirCQrVXF/KoFQ9r9jnKd+KRA5P1YVsq'
    '2FL+Gc5HNBz5qpS4WC0u5HJyVMQtIh6dqcBfsm7FkxAWqqYjbt5SzTM8Xzm+uHUHn/kZFXI5HgUmLiI+7mM/2Zu43fgXD+Ev4eDEhSXOQFYO6VHy1ImP64nb'
    'k9WyrHYWUlv3+kjR1Ofz8fvy2QDZyS7J15Nku04pZQe68vCMTsWXxPXIkjgqqjAENNiuxXgQsTlP+rBoemYud+PGRScJIslza10utynUoxoOQCpC5XfPx2bb'
    'Jm1UwyivvLKatPOUjtcT+BAm7mNjFDZd4xOuXOsUriW1OpbPx6raSXZ/FPH4LB6EYhPGzM/Pi2v/4/rNhUUYioXMi5fvzFq+NXv37VnLFKk7i8kBMBUOVDUw'
    'sOUIsZkwd0g4MNknN4CGcYAPH+D0AWICGFLgjn0qwE2NePmql+z4uNnxEXMqOgLjyPlD7l4xfYbRH31MqU0OHiN3TAZ4HxbVyyYty3Yzl5sXqCCiBsRdVrsp'
    'n1R9FdSXxNvdNj1EVo9RHZo3QmYSumBW1V4kWzVByNF346EffawLsR1PJthQkv1I7Re5fP1giU4N4Ohx4iKWrbr8qYYgvGmAGPAe+Mp9gyoUKGcgWlYaBC0s'
    '0H1gINXEC7gC+La3EDBzlZ/vOupoS3BRRHQjnEvcn5coU4h7st+gmIMBBGcORwl10FDDn+ks2Z8I6dX5tqCoWodEHerUm+61Dr9zr7UQ5dlrcCVcn8zcO3II'
    'wmXTUBt8rGoWU5vZACaSUaTaUYZIdBSF4TJjos05x4noQoae8jh29IB9x0SBqlMDI0sTm4QA5XRAZKnP+hTdASg+NoemGpJSTfeStH4zYdFI4wQzHOh+PwB9'
    '4/6yx8VPTeLQ0/WiXpSMvbizBtrFNhPrP6FKrpbNbfBl2mcc7lrtMBth9lXIYTnxJhRMFf6qg7xNmI/HE9gGb4D+OvcFN5LVU4HWpg563OM8m0eUunoYTrku'
    'zQ/1qMpxJgM6nwx25KdyiBDXahItCLTp6PBywqk0bSHJoUNf02tcoJevGcq/YEEG9zr8Js82XX8SORk9AYdUp8jHmtNAACc+eW1aKioYVUsdRW8Nn8IB7jLZ'
    'lmRCyh5gPcVdtQ9K4gaDjxQ36i/oUmV69KYtgP8Jlq66Pc9OktixbKe/UoVFZeWWTUWbFDBT2pSVJiTOTra4PIAsdA3ycyJfnrHewkcTu8zdAJBqAq2vJ9Bw'
    'Vvc4HtEy/Odim5Iruj60A9FF8gjEDHbd9nUPQBUQ8/PHLGPwiqqcorR/KzZklaCOAoFfkEpV3+bht6LLKU/cY1XlLpl4TtqDNkkqofSZSEoyALrhBN4F1AGe'
    'TOvDf09bLZOEyMB2yr2hHw/7VCYQoWCQdPPvywhcWf1h5fsHK/dXscR/kISvJ8ZWOqnWZe5jFTfFIvci15HV/ZQhyFhngP5mDGRwnzQIcNla5Guz5KCO1b4P'
    'dS6hBtSRw45fat2bZgupDR0nbVOmGNjrKV+alDKAfgf0JJzwmUYMsBQUxFJmgOBxgqUhu8QzBPf7CutPPV5wXDqO2iW9zwQJvG1vzRoXCJSpWmzAXT7gHcMI'
    'BQ2ZRVSQa4oYnZ6g+R80OPkGRZocqIF2EQLEAVLnoEeGUAa1xqHc2MTTPNEqGewRObtv+CTufUZcp31M6ybdzM7HEDwIcsuxWdJBo7tNQ90tIwY4+UODLUhA'
    'G19G2JIobHyVv7tS+Pbvaysbd/OF+xuLN5dX1tcX1v9F4bA2oj1oxaHt0Vai9u7OffMA94m3j57O0tfXDhhaeMvDgJQXqXoBSUq8jPjSkgv1IfTzFnjMRJ2a'
    'yULLf9PIaQLQ64tmAGFkQ4nt1M1MwEkyUwjH0sqZXVKNiyI1haTOZyuf0VN5OBr76nTfIIQVxEHXCj42gY95zOJiOmPZkYjxAQLCGqCn2gOjKVGlukgtr2gK'
    'RYfY8fOaVnGLEVjT7MEoFO8nq4XC2j8JBVzWuqrpBPZIUzbcNTrrau5v29wzWP7vhmmT2bT+TLOGkIK40qR86uEu3YwtGrgV5jVjp+RouDGTAAzTex6r4ZMG'
    '2Q8TwH0XRlXCpGYhzG3voeDruEysYBk6jUOKTGaoK4MzASmFLMLEcX3WodM15iwCDZRsLyhDm1nc6tEwc/qlWkSnhGD0u5QTwnRavfYCn1btyJfpE2YMo7Qp'
    'aZzhMLB8kyWPNg3vRztIGtKAmaGFpEZmB88obV/90k+fEbGPz4GUdCQ1rHABY1d73vRnEv2LiOxF4t7KP1aXCyv31r9fLdBZ335HEQcIMjl/UdJiugFk8Q8Z'
    'l8YPVkY/nYHv8rpEbuus6NZGJctixbY0ONvqa+zMqLr3TdVhI059F+POiY3V9bWN+wWziv+o+GW9uHDv7pxYe3B//QGu43l5XFv4qvDDnPh6beNeIc+/zi3j'
    'wjJ/x3aqLc0MPCBr2feipJUsz6WaZ3Sjph8EyFWtt6lt7oWoS1n3rENWzJKsm7T1yITkImdYsEfoA1mbafnGrYOAn7TKIDkqpMsx+i9QSwMEFAAAAAgAAAAh'
    'XLUDnItQCgAAjBoAABMAAABTVFIwMV9NMDhfbGVzc29uLm1k7VlRTxvZFX73r7hStFIruYBJUu3ysoqaVRW12a42fWpVgTfQ3ahJQZis1DcDQ+Rg0hjwEJuM'
    '3WHXJHjX7Q7GBFtxXvJTNm9z7/yHfuecO+NxQuhr1a6EDB6P7z3nO9/5zneHS+rW7z+fyqmbUx+qH4tVZXxHt3t4VTrw8KP0d0emUVN85UVRbxWVPumF/YfR'
    'Tknpx3VT9hTehUFFmc1WtNnXfgl/KFzX/+rQm0bPbNQymUuXlFlfjert8GSIRdvRtpfJhEERF9VvPruBxR3jNJQu93R5YJoVfeAZx7N76hNHmWYnPPFfn5kH'
    'W4jp9Zmsoa7/7papF1XYDSL3EX69Qnj6cGjXi9y2Lpd0uTWRyoDz4VWwv03CPKkovVnDDjY/xK1MyzF7Pb3r6c2qsrGaF+2w2+b81mr6WcfUWgjVbqzMk6p5'
    '7vAGF6BhGkinVIv2HkauR8EJPE8r+qRvP1C6WzWeQ6lFjwFvMZP5RXKH40frgOqHPtbixEwXW5dbZvMUeQA2ijfeIAYg+T4gXC+pX99ZUdHf+8Y5VpcRXhg4'
    'b4pVWqN/EAY7nL3r6HLtTdFN7cxrzlj0CObcxFUdBFk1za+X8UpoRGstsx+MwNT/eMg5+yUd1Iw3pHsI3EaPI2VI6YbctGkMEQqD7buR27Nl54oiOGLbiWPW'
    'Agsc0VM/6uvnW8rs9pV2twgxXT4K+4GNEvQMsCvSKRFSzUMzqPFyj6rRXol2pW+hYraExHt/oGj7uqsfcwkAdbRe0Yc+3z5wwtNK5BJZspbU0VZF5aL9LQR+'
    'bMq+EkTp7qtTdBkB6JNTBXA5LYndpo/FjHdg9tsxxSQn+rQdcFl9oNXcJgIj+Wj/EbdQpxf2iojC1vaPwoo/nUdQNMuBp6ttG5biTqdGwD0nxG81/vm06ZU4'
    'urVe9n2wqI+ufkD3IGCJFFA0amatg07ETco8L4I03FbPm2F3K+kQz0ES1Hi82arxVy0h0LH6uw2Agw8nUpxbC8wLT3eGM6qwfHtyPl/46ovF/PL8ZGFleSo3'
    'm19amlj6m/rZrZXlhfy9u3dWJq/jDqV3j38uXeV7ISEPRfl2i5oI76IHpwwouA9WIMvXZ79avJv/gsCm3osvSkspvdGnVn6xGl+Jqg7Jk9/AZfWHG59xtK+q'
    'AHJGzb0/yLkJFbmnptzBryO0ErEwPO1ENQcL+0VQU0ilTPf7OJToqaOu3SAOmhdVlquTIpAca2v9KEAJIieYUWORzc3nV/KTn39y7frNTybuzc8x64+G2Aql'
    'eoU87BJUp+bhRyHUPkLP73rgS9U1G1XtlECgXG7sEyYn4NvQ/mHul/gIwUtoiBUV/hCXbGao6Lj+QGeb28rsbZtyA1Dx6JnF6Jn98+LyvQLFCJ49VvOLtwuT'
    'ywtLi8srBepMUrq6w+WSqkVlMETKC+51S1YL6V6LsOBK9Y620N3DiwtDt3FJcNvC1/m79/MrC3PMFce4r2bUfPbL+/iS+vHBjroCakHSRp3w+iwRf2RcTH+Q'
    '6AYVmFUaI8sKKV1ydflUyXoTDEupZgVPmVqJpBidTin+0LfXpxS63gQBCxFKsV2RvjmiTosed5BGTCIrHHuE2FRaworq0/yntCqJUHjmmUHddicDuH+EXZlx'
    'HEIiRwjNOdabh7R1XFwvpqDoW8JEyGKDCyVEJ5G3fchtQuxJxrJch4wQJkIzwhAbEo1M0DcHAU2CmzmKAdW5A14vzN5dmP9yYXnUQ2NMG2MBNxH3lHPMe3Fr'
    'MYJI+aBnAcOcwJvxPlMEhc98judpjwYurRjPgZHDEMSv3SBtjepV1P8EuPWB8TA8DoBxMXpAVIBlOKzRgiyU0FBPBtzaUThYjZMgdkdunQQoqntwKXTlkv07'
    'xxI+mgbQXRoPGEOYfZmMlU6xSOAnJiCRjrHH/qyFnFPjlc3cTitYIHQ810sKlfgPwtLehKKGLxE/rNHeS5L2WJP8cLD1Ma0rcaSggYTxJDgI+31rbeI4qA88'
    'dBlqX3cJLR20ddcVv+CmqppKf5rTt7rIgdoYYClRxVZ8lXPkasc17sKVDBlsW3p8c6dll8J3gaflN4hDMdN82m+nUhkjxkw8bkcQsocRG6amr+Jvnm2MK0Uw'
    'hibNUYsmRh/kkemVdNk7DHuv0JwjStLwLJ8sEGlNFixYSGK9mQqPexyO1ZDzYL8srBM2k384CLgCzY6ADOJZTo/QH5tr8Y3pphF1E/+uK/B3xz3JfUgIJglx'
    'zR6QMcXmcDbjWFAYFgNmaG2IcLErXAim+aZxSYOA+8fUncZpAW3wTpgjxuky9k38jKMSbnK8bLg4X5sJsaeOIMunoAPBZs2pUPh89K7IAevUM99s2A6R1rX+'
    'JmnbmCLMZBIRJpxVnaSJzxGbHkIj98+4e1h17PjjwoUjzg74leWByWYUf3b6jDSPVooInAM3cIIZUDuSwV0fm+i98SS4WudIVw/nL5qNRfOsocy3Q7KHfgVZ'
    'guZPajSGILUENzqM9hG3jhOE8RgYOTrAQH8AY1zBx2NTI7ahyfRjK/0OpZkN8VhkPw2+7bZo99jJyJST2ZCce0q6MYxno+myjx7t/lYbot0HONUckSVOEXH0'
    'Ae9WbulnyQpyKnjrsACprQ6xmR2eNCgoNPx+vg2pR4VRqSY8dc/ewc0Nu/eyk6whZ4anMOrgwcDhw6c4jzGLn+RZT3J6ewa0e+pe/i8Ls4X8vaW7C+zDvrpD'
    'iFPXkzBLS3+zgT6xks4kwaw5aIx3gGUjs73u0P3CbbJn/8GWzVkbaD/A60KhMCsXyS/SLYv3V5bu4xYYSbjGwsTtwtd0nT2lpTXaKpuaz2a/arr/BLCNbbRw'
    'lvncK+F9Wr6ysQcHcuEZjs140c+GbOVG8z99UpEWk9zokYVMGz5AMkt5EgNl/ObzbZZ9Duror9pesuJKi0oXsu2SuMU+pPoRW5Aw0tOHt0AHszzuKUab9YCb'
    '2Z60YAp41iTymBwFMhlrq7nB2b7UrHxRQnZCx9XLTfBzI7EhM//VT1jYHrsOCk8jX04vo3KQirEjgY2SczgmMB9X/AqxNj4TkIQ1N0iu/UN7OOCdeKAko4E8'
    'n5yFiSlc19dncBieY5pOrIgCGnsNKd5eyY6ozLSFlT3IRadeQiEODUVOz1oJ7k2x+tPR5aKjy5siGU0wzhwcR64cVLrf21nA3c/PTrj9kf+kSAC+Yu1jUjRK'
    'FaLwW6jT4l+p+HyKkNFNO3BG1lNjVgU1U1tNlfyyLTlTY+Z/7WGOLWjSbVYMMUgwoUV1BzV8C72SmiysyZMiycmRzVriFO4/uYn/IzfBwxtWsnk4ObLt2EbG'
    'JnWqPWOLQaczZnzkGDXbFWk2GY4X6WtWXew+suoc65EV4zH57sMtainRAfYkCI6nbBjsIF428aIqNCgGJLKcoTwfoNn4JEA76kpylIEuws6zVIuXl5qiVLgQ'
    'byEb8vTHHoQnDQrie9QsWSMwgiaT+hfBBf8hmJQF5R8Fyf8JJuwjFDkaI1t5CgPhwVCVi9QU8vRhEyX2oSspp/hvUEsBAhQDFAAAAAgAAAAhXKfdOsyNCAAA'
    'WxQAABIAAAAAAAAAAAAAAKQBAAAAAFNUUjAxX00wOF9mb3Jtcy5tZFBLAQIUAxQAAAAIAAAAIVy1A5yLUAoAAIwaAAATAAAAAAAAAAAAAACkAb0IAABTVFIw'
    'MV9NMDhfbGVzc29uLm1kUEsFBgAAAAACAAIAgQAAAD4TAAAAAA=='
)
_EMBEDDED_ZIP_BYTES = _embedded_b64.b64decode(_EMBEDDED_ZIP_B64)
print('📦 노트북에 내장된 실습자료를 준비합니다. ZIP 업로드는 필요하지 않습니다.')
_prepare_project_zip(_EMBEDDED_ZIP_BYTES)


## 미션 핵심 개념과 원본 실습 설명

---

## 🎯 미션 08. 전면전 발발 모의 전 부대 보급품 동시 급감 스트레스 시뮬레이션

**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침

#### 미션 배경 및 당위성

- 전면전 모의 전 부대 보급품 동시 급감 스트레스 시뮬레이션을 수행한다.
- 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **전 부대의 소모가 동시에 늘고 보급이 줄어드는 가상 충격 시나리오로 경보 엔진의 스트레스 시뮬레이션을 수행한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.
- 이 문제를 해결하기 위해 본 미션에서는 **소모를 1.5배, 2배, 3배로 키우고 보급량을 절반으로 줄이는 충격을 12일간 가정해 재고를 다시 산출하고 모델에 적용하는** 실습을 체계적으로 수행합니다.
- 학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/dashboard/str01_app.py (Streamlit/Dash 등)` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)

#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 의사결정 지원에 최적화된 시각적 위계 및 사용자 경험(UX)

#### 미션 PASS 조건

- 정량 기준: 대시보드 로딩 시간 2초 이내, 경보 발령 정확도 95% 이상

- 진행 기준: 웹 기반 분석 대시보드 구동 및 실시간 필터링 기능 검증

- 검증 방법: 실제 군수 데이터 연동 화면 테스트 및 UI 동작 검증

#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.


### AI 활용 예시 · 미션 08 전용

> 나는 Python을 처음 배우는 학습자야. **전면전 발발 모의 전 부대 보급품 동시 급감 스트레스 시뮬레이션**을 진행하고 있어. 다음 요구사항을 보고 핵심 함수의 입력·출력, 필요한 검증 3개를 쉬운 말로 설명해줘. `TODO` 구간만 구현할 수 있는 코드를 제안하고, 데이터를 미리 보지 않는 검증 방법과 경계 사례도 알려줘. 실행 결과는 만들지 말고 내가 실제 실행한 로그를 받은 다음 분석해줘.  
> 요구사항: src/dashboard/str01_app.py / evaluate: d,guard → 4등급 정확도·경보 여부 정확도·미탐률·전체 표본 수·정답 등급. 양수 재고 예측과 이미 재고0 관찰을 모두 분모에 포함하고 실제 양성0 미탐률은 NaN이다. 규칙 경보의 효과와 예측 모델의 성능을 구분한다.

**AI 답변을 받으면:** 그대로 복사하지 말고 입력 시점·자료 범위·실제 실행 여부를 직접 확인하세요. 확인되지 않은 수치는 `미측정`, 수행하지 않은 활동은 `미실시`로 남깁니다.


In [ ]:
from pathlib import Path

if not (globals().get('STR01_PROJECT_READY', False) and globals().get('STR01_PROJECT_DIR') and Path(STR01_PROJECT_DIR).is_dir()):
    print('먼저 ① 실습자료 자동 준비 및 환경 설정 셀을 실행한 뒤 이 셀을 다시 실행하세요. ZIP 업로드는 필요 없습니다.')
else:
    """전 부대 소모가 동시에 크게 늘고 보급이 차질을 빚는 가상 충격 시나리오(소모 1.5·2·3배, 보급 50% 감소)를 시뮬레이션하고, 경보 엔진의 등급 정확도와 대시보드 갱신(로딩) 시간을 측정하며 예측 모델에 DOS 안전장치를 더한 보강 효과를 확인한다."""
    import time
    import numpy as np
    import pandas as pd
    from sklearn.ensemble import GradientBoostingRegressor

    # ---------------------------------------------------------------------------
    # [교육용가상자료] 가상 부대 20개 x 핵심 보급품 5종 일일 재고·소모 시뮬레이터 (실제 부대·수량·기준일수 아님)
    # 핵심성과지표(KPI) = 기준 보급일수(DOS) = 현재 재고 / 최근 평균 일일 소모량. 목표 DOS는 교육용 가정값이다.
    # 결핍 위험 일수 H = 보급이 없다고 가정할 때 현재 재고가 바닥나기까지 남은 일수(미래 실제 소모량으로 계산한 정답).
    # ---------------------------------------------------------------------------
    ITEMS = {"식량": (100, 0.3, 10), "피복": (20, 0.1, 15), "유류": (300, 1.2, 7), "탄약": (60, 1.5, 6), "수리부속": (15, 0.5, 12)}   # (기본 일소모, 템포 민감도, 목표 DOS)
    N_UNIT, DAYS = 20, 160
    UNITS = [f"가상{u:02d}부대" for u in range(1, N_UNIT + 1)]


    def make_data(seed=1, n_unit=N_UNIT, days=DAYS):
        """일 단위 재고 원장: unit, item, day, tempo(실제 작전 템포), plan(사전 계획 템포), wx(악천후), consume, resupply, stock."""
        rng = np.random.default_rng(seed)
        rows = []
        for u in range(1, n_unit + 1):
            s, ph = rng.uniform(0.7, 1.5), rng.uniform(0, 6.28)
            surge = np.zeros(days)
            for st in rng.choice(days - 10, 4, replace=False):
                surge[st:st + int(rng.integers(5, 10))] = rng.uniform(0.4, 0.8)
            tempo = 1 + 0.25 * np.sin(2 * np.pi * np.arange(days) / 30 + ph) + surge
            plan = tempo + rng.normal(0, 0.08, days)
            wx = (rng.random(days) < 0.15).astype(int)
            for it, (b, sens, tg) in ITEMS.items():
                stock, off = tg * b * s * rng.uniform(0.8, 1.2), int(rng.integers(0, 4))
                for t in range(days):
                    c = b * s * (1 + sens * (tempo[t] - 1)) * (1 + 0.12 * wx[t]) * rng.normal(1, 0.08)
                    r = max(tg * b * s - stock, 0) * rng.uniform(0.9, 1.1) * (0.5 if rng.random() < 0.08 else 1) if (t % 4 == off and t > 0) else 0.0   # 4일 주기 정기 보급: 목표 재고까지 채움(가끔 수송 차질로 절반)
                    stock = max(stock + r - c, 0.0)
                    rows.append((f"가상{u:02d}부대", it, t, tempo[t], plan[t], wx[t], c, r, stock))
        return pd.DataFrame(rows, columns=["unit", "item", "day", "tempo", "plan", "wx", "consume", "resupply", "stock"])


    def make_samples(d, lo=21, hi=125):
        """(부대, 품목, 기준일 t0)별 특성(피처)과 정답 H. 피처는 t0 시점까지 알 수 있는 값과 사전 계획·기상 예보만 사용한다."""
        out = []
        for (u, it), g in d.groupby(["unit", "item"]):
            c, stk, rs = g["consume"].values, g["stock"].values, g["resupply"].values
            pl, wx, tp = g["plan"].values, g["wx"].values, g["tempo"].values
            cs = np.cumsum(c)
            tg = ITEMS[it][2] * ITEMS[it][0]
            for t0 in range(lo, hi):
                idx = np.searchsorted(cs, cs[t0] + stk[t0])
                if idx >= len(cs) or idx <= t0:
                    continue
                c3, c7, c14 = c[t0 - 2:t0 + 1].mean(), c[t0 - 6:t0 + 1].mean(), c[t0 - 13:t0 + 1].mean()
                last = [k for k in range(t0, t0 - 15, -1) if rs[k] > 0]
                out.append({"unit": u, "item": it, "t0": t0, "H": idx - t0, "stock": stk[t0], "c3": c3, "c7": c7, "c14": c14,
                            "dos7": stk[t0] / c7, "dos3": stk[t0] / c3, "trend": c7 / c14, "burst": c3 / c7,
                            "plan7": pl[t0 + 1:t0 + 8].mean(), "plan_ratio": pl[t0 + 1:t0 + 8].mean() / tp[t0], "tempo0": tp[t0],
                            "wx_fc": wx[t0 + 1:t0 + 8].mean(), "since_rs": (t0 - last[0]) if last else 15,
                            "rs_avg": rs[t0 - 13:t0 + 1].sum() / 14, "fill": stk[t0] / tg, "item_code": list(ITEMS).index(it)})
        return pd.DataFrame(out)


    def mape(a, b):
        # [TODO : 학생 실습 8] 전면전 발발 모의 전 부대 보급품 동시 급감 스트레스 시뮬레이션 관련 핵심 로직을 구현하세요.
        # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
        # <<< 여기에 코드를 작성하세요 >>>
        pass  # 작성 후 pass를 제거하세요.
    d0 = make_data()
    s0 = make_samples(d0)
    X = [c for c in s0.columns if c not in ("unit", "item", "t0", "H")]
    gbr = GradientBoostingRegressor(n_estimators=150, max_depth=3, learning_rate=0.08, random_state=0).fit(s0[s0["t0"] <= 90][X], s0[s0["t0"] <= 90]["H"])
    T_SHOCK = 100
    def level(h):
        # [TODO : 학생 실습 8] 전면전 발발 모의 전 부대 보급품 동시 급감 스트레스 시뮬레이션 관련 핵심 로직을 구현하세요.
        # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
        # <<< 여기에 코드를 작성하세요 >>>
        pass  # 작성 후 pass를 제거하세요.
    NAME = np.array(["정상", "주의", "경계", "심각"])

    def shock(d, mult, rs_cut=0.5, span=12):
        """T_SHOCK일부터 span일 동안 모든 부대의 소모를 mult배, 보급량을 rs_cut배로 바꾸고 재고를 다시 계산한다(가상 충격 시나리오)."""
        d = d.copy()
        m = (d["day"] >= T_SHOCK) & (d["day"] < T_SHOCK + span)
        d.loc[m, "consume"] *= mult; d.loc[m, "resupply"] *= rs_cut
        for (u, it), g in d.groupby(["unit", "item"]):
            idx, st = g.index, g["stock"].values[T_SHOCK - 1]
            new = []
            for t in range(T_SHOCK, DAYS):
                st = max(st + d.loc[idx[t], "resupply"] - d.loc[idx[t], "consume"], 0.0); new.append(st)
            d.loc[idx[T_SHOCK:], "stock"] = new
        return d

    def predict(s, guard):
        p = np.clip(gbr.predict(s[X]), 1, None)
        if guard:                                                   # 안전장치: 전 부대 소모가 동시에 급증(충격 모드)한 날에는 최근 3일 소모 기준 DOS가 더 짧으면 그 값을 채택
            share = s.groupby("t0")["burst"].transform(lambda b: (b > 1.15).mean())
            p = np.where(share > 0.4, np.minimum(p, np.clip(s["dos3"].values * 1.05, 1, None)), p)
        return p

    def evaluate(d, guard):
        s = make_samples(d, lo=T_SHOCK + 1, hi=T_SHOCK + 14)
        lt, lp = level(s["H"].values), level(predict(s, guard))
        acc = float((lt == lp).mean())                                              # 4등급 일치율
        alert = float(((lt >= 1) == (lp >= 1)).mean())                              # 경보 발령(주의 이상) 여부 일치율
        miss = float(((lt >= 2) & (lp < 2)).sum() / max((lt >= 2).sum(), 1))        # 경계 이상 미탐률
        return acc, alert, miss, len(s), lt

    print("[충격 시나리오 시험] 전 부대·전 품목 동시 소모 증가 + 보급 감소(12일간, 가상 모델) [교육용가상자료]")
    rows = []
    for mult in (1.0, 1.5, 2.0, 3.0):
        d = d0 if mult == 1.0 else shock(d0, mult)
        a0, al0, m0, n, lt = evaluate(d, False); a1, al1, m1, _, _ = evaluate(d, True)
        rows.append({"소모 배율": f"{mult}배", "표본": n, "경계 이상": f"{(lt >= 2).mean():.0%}", "모델만 경보정확도": f"{al0:.1%}", "모델만 미탐": f"{m0:.1%}",
                     "보강 경보정확도": f"{al1:.1%}", "보강 미탐": f"{m1:.1%}", "보강 4등급": f"{a1:.1%}"})
    res = pd.DataFrame(rows)
    print(res.to_string(index=False))
    acc_guard = float(res["보강 경보정확도"].str.rstrip("%").astype(float).min()) / 100

    # 대시보드 갱신(로딩) 시간: 한 시점의 전체 상황판(100건 예측 + 등급 집계 + 문자열 생성)을 50회 반복 측정
    d3 = shock(d0, 2.0)
    times = []
    for _ in range(50):
        t0 = time.perf_counter()
        s = make_samples(d3, lo=T_SHOCK + 5, hi=T_SHOCK + 6)
        p = predict(s, True)
        lv = pd.Series(NAME[level(p)]).value_counts().reindex(NAME, fill_value=0)
        board = " / ".join(f"{k} {v}" for k, v in lv.items())
        times.append(time.perf_counter() - t0)
    times = np.array(times)
    print(f"\n상황판 1회 갱신: 평균 {times.mean() * 1000:.0f}ms, 95백분위 {np.percentile(times, 95) * 1000:.0f}ms, 최대 {times.max() * 1000:.0f}ms (목표 2초 이내, 환경에 따라 다름)")
    print("예시 상황판(소모 2배 충격 105일차):", board)
    d_n = evaluate(d0, True)[1]
    print(f"\n충격 규모를 키울수록 경보 대상이 늘고 모델만 쓰면 미탐이 늘어나므로, 단기 소모 기반 안전장치로 보강했다(정상 시 경보 정확도 {d_n:.1%}).")
    ok = acc_guard >= 0.95 and times.max() < 2.0
    print(f"목표(대시보드 로딩 2초 이내, 경보 발령 정확도 95% 이상): 로딩 {'달성' if times.max() < 2 else '미달'}, 최저 경보 발령 정확도 {acc_guard:.1%} {'달성' if acc_guard >= 0.95 else '미달'}")


## 📋 3. 학습활동 4단계

### 활동 1: 무엇을 확인하고 계획하나요?

**이번 활동의 과제:**

#### 1. 단계 개요

- **목표:**
  - 원본 목표, 기본 코드, 그리고 제공된 학습 자료를 면밀히 분석하여, "충격 기간을 늘리고 평가 구간을 변경할 때 어떤 요소를 고려해야 하는가?"라는 핵심 질문에 대해 사전에 엄밀히 판단합니다.
  - 아울러 전 부대의 소모가 동시에 늘고 보급이 줄어드는 가상 충격 시나리오를 바탕으로 경보 엔진의 스트레스 시뮬레이션을 수행하는 과정에서 AI 에이전트에 위임할 분석 범위, 데이터 제약 사항, 입력 및 기간·단위·사용 가능 시점, 그리고 evaluate 함수의 입출력 계약 조건을 체계적으로 수립합니다.
- **수행 방식:**
  - 막연한 코딩이나 AI의 무분별한 답변에 의존하지 않고, 전시 상황과 같은 극한의 자원 부족 상황에서 경보 엔진의 신뢰성을 검증하기 위한 명확한 사전 규칙과 검증 계획을 문서화합니다.

#### 2. 세부 수행 절차

- **원본 목표·기본 코드·자료 분석 및 핵심 질문 검토:**
  - 전 부대의 소모 증가와 보급 50% 감소라는 스트레스 시나리오에서 경보 엔진의 성능을 평가하기 위해 필요한 입력 데이터, 기간, 단위, 그리고 사용 가능한 시점의 제약 조건을 파악합니다.
  - 시뮬레이션 기간을 확대(예: 12일 → 25일)하고 평가 구간을 변경할 때 발생할 수 있는 데이터 왜곡이나 샘플 생성( make_samples 의 hi 조정 등)상의 제약을 사전에 분석하고 정의합니다.
- **evaluate 입출력 계약 정의:**
  - 파라미터 d 와 guard 등을 입력받아 4등급 정확도, 경보 여부 정확도, 미탐률, 전체 표본 수, 정답 등급 등의 평가 지표를 산출하는 핵심 함수 evaluate 의 입출력 계약 조건을 명확히 이해합니다.
  - (단, 양수 재고 예측과 이미 재고 0인 관찰 건을 모두 분모에 포함하되, 실제 양성 0에 대한 미탐률은 NaN 으로 처리됨을 인지합니다.) AI 위임 범위 및 검증 계획 정의: AI …

#### 완료 확인:

- 이 활동의 확인 근거 M08 계획: 전 부대의 소모가 동시에 늘고 보급이 줄어드는 가상 충격 시나리오로 경보 엔진의 스트레스 시뮬레이션을 수행한다.에 필요한 입력·기간·단위·사용 가능 시점과 evaluate의입출력 계약을 설명하고 본인의사전판단·AI위임·검증계획을 기록했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

#### 내가 기록할 것:

- 직접 확인한 입력·원본 조건:
- AI에 실제로 요청한 질문과 받은 답변의 위치:
- 실제 실행 결과/오류 메시지:
- 잘못된 설명, 수정한 코드, 수정 이유:

### 활동 2: AI와 코드 초안을 만들어 실행하기

**이번 활동의 과제:**

#### 1. 단계 개요

- **목표:**
  - 핵심 함수인 evaluate 를 구현·저장하고 첫 성공 실행 결과를 온전히 보존합니다.
  - 파라미터 d 와 guard 를 입력받아 4등급 정확도, 경보 여부 정확도, 미탐률, 전체 표본 수, 정답 등급 등의 평가 지표를 산출하며, 양수 재고 예측과 이미 재고 0인 관찰 건을 모두 분모에 포함하되 실제 양성 0에 대한 미탐률은 NaN 으로 처리하는 계약 조건을 준수합니다.
  - 아울러 규칙 경보의 효과와 예측 모델의 성능을 명확히 구분하고, 변경 과제(충격 기간을 12일에서 25일로 늘리고 평가 구간을 함께 명시하여 성능 비교, make_samples 의 hi 조정 등)를 수행하여 실제 코드와 출력 결과를 정확하게 확보합니다.
- **수행 방식:**
  - AI와 협업하여 스트레스 시뮬레이션 및 평가 로직을 코드에 반영하고, 기준 코드 실행 및 변경 과제 실험을 수행하여 신뢰성 있는 성능 비교 데이터를 구축합니다.

#### 2. 세부 수행 절차

+ **코드 구현 및 첫 실행**
  - 핵심 함수 evaluate 구현 및 첫 성공 실행: src/dashboard/str01_app.py 내에 evaluate 함수를 구현하여 저장하고 첫 성공 실행을 보존합니다.
  - d 와 guard 인자를 통해 4등급 정확도, 경보 여부 정확도, 미탐률, 전체 표본 수, 정답 등급을 산출합니다.
  - 양수 재고 예측과 이미 재고가 0이 된 관찰 건을 모두 분모에 포함하되, 실제 양성 0에 대한 미탐률은 NaN 으로 처리하는 계약 조건을 정확히 반영합니다.
  - 규칙에 따른 경보 효과와 예측 모델의 성능을 명확히 구분하고, 실행 로그와 표·그림 형태의 출력을 보존합니다.

+ **변경 과제와 전후 비교**
  - 변경 과제 수행 (충격 기간 확대 및 평가 구간 명시): 충격 시뮬레이션 기간을 기존 12일에서 25일로 연장하고, 평가 구간을 함께 명시하여 모델 성능을 비교합니다.
  - 기간을 늘릴 때 발생하는 샘플 생성 함수…

#### 완료 확인:

- 이 활동의 확인 근거 M08 구현: src/dashboard/str01_app.py의 evaluate를저장하고계약〔d,guard → 4등급 정확도·경보 여부 정확도·미탐률·전체 표본 수·정답 등급.
- 양수 재고 예측과 이미 재고0 관찰을 모두 분모에 포함하고 실제 양성0 미탐률은 NaN이다.
- 규칙 경보의 효과와 예측 모델의 성능을 구분한다.〕을충족하는 첫성공실행·로그·표/그림을보존했는가?
- 이후Lesson의추가변경을실제코드에반영했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

#### 내가 기록할 것:

- 직접 확인한 입력·원본 조건:
- AI에 실제로 요청한 질문과 받은 답변의 위치:
- 실제 실행 결과/오류 메시지:
- 잘못된 설명, 수정한 코드, 수정 이유:

### 활동 3: 원자료와 결과를 대조·수정하기

**이번 활동의 과제:**

#### 1. 단계 개요

- **목표:**
  - 앞서 구현 및 수정된 스트레스 시뮬레이션 및 대시보드 코드를 저장하고 재실행합니다.
  - 대시보드 로딩 시간 2초 이내, 경보 발령 정확도 95% 이상을 달성하며, 이미 재고가 0이 되어 결핍된 관찰 건을 분모에 포함하되 이는 모델 예측 성공이 아닌 규칙 처리임을 명확히 구분합니다.
  - 아울러 경보 여부 일치율(95%)과 4등급 일치율의 차이를 엄격히 판정하고, 50회 갱신 시간 측정 시 피처 생성·예측·집계·문자열 생성의 로컬 시간을 기준으로 삼으며, 기간 연동 시 make_samples 의 hi 조정 및 종료 범위를 기록합니다.
  - 아울러 AI의 설명 및 코드 선택 3건 이상을 실제 군수 도메인 자료 및 실행 결과와 철저히 대조하여 보완된 답변을 도출합니다.
- **수행 방식:**
  - 단편적인 실행에 그치지 않고, 실제 기간, 분모, 조건, 수치를 바탕으로 최초 실행 결과와 수정 실행 결과를 비교 대조하며 AI의 답변을 비판적으로 교차 검증합니다.

#### 2. 세부 수행 절차

- **수정 코드 저장 및 재실행:**
  - 대시보드 및 스트레스 시뮬레이션 소스 코드( src/dashboard/str01_app.py )를 안전하게 저장하고 재실행하여 에러 여부와 실행 로그를 확보합니다.
- **정량적 성능 지표 및 시각적 위계 검증:**
  - 대시보드 로딩 시간을 2초 이내로 유지하고, 경보 발령 정확도 95% 이상을 달성하며, 의사결정 지원에 적합하도록 경보 위계가 시각적으로 분명하게 표현되는지 확인합니다.
  - 추상적인 소모 배율과 보급 50% 감소라는 시험 조건 아래에서, 정확도 분모에 관찰 재고 0건을 올바르게 포함하고 이들이 모델 예측이 아닌 규칙 처리에 해당함을 엄밀히 검증합니다.
- **지표 간 차이 및 기간 연동 제약 사항 검토:**
  - 경보 여부 일치율(95% 이상)과 4등급 일치율을 혼동하지 않고 각각 분리하여 판정합니다.
  - 50회…

#### 완료 확인:

- **판정에 필요한 근거**
  - 이 활동의 확인 근거 M08 검증: [선택] 충격 시나리오·로딩 시간 — 대시보드 로딩 시간 2초 이내, 경보 발령 정확도 95% 이상이며, 의사결정 지원에 맞게 경보 위계가 시각적으로 분명한가.
  - 실제기간·분모·조건·수치로판정하고 최초/수정실행을비교했는가?
- **미션별 확인 사항**
  - 시험은 추상적인 소모 배율과 보급 50% 감소입니다.
  - 정확도 분모에는 관찰 재고 0건을 포함하며 이들은 모델 예측 성공이 아니라 규칙 처리입니다.
  - 경보 여부 일치율 95%와 4등급 일치율은 다릅니다.
  - 50회 갱신 시간은 피처 생성·예측·집계·문자열 생성의 로컬 시간이며 웹 서비스 전체 로딩 시간이 아닙니다.
  - 기간을 늘리면 make_samples의 hi도 조정하고 종료 범위를 기록합니다.
- **AI 검토 기록**
  - AI 주장/선택3건을근거로검토·보완·재실행했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해…

#### 내가 기록할 것:

- 직접 확인한 입력·원본 조건:
- AI에 실제로 요청한 질문과 받은 답변의 위치:
- 실제 실행 결과/오류 메시지:
- 잘못된 설명, 수정한 코드, 수정 이유:

### 활동 4: 최종 결과와 수행 상태 기록하기

**이번 활동의 과제:**

#### 1. 단계 개요

- **목표:**
  - 미션 8(전면전 발발 모의 전 부대 보급품 동시 급감 스트레스 시뮬레이션) 전 과정을 수행하며 산출된 대시보드 소스 코드 , 스트레스 시뮬레이션 분석 보고서 , 결과 CSV 파일 , 그리고 실습 기록 서식 을 누락 없이 집대성합니다.
  - 아울러 정량적 목표 달성 여부, 미측정 영역, 문서 작성 상태 및 교관(또는 시스템) 승인 대기 상태를 명확히 구분하여 최종 판단을 기록합니다.
- **수행 방식:**
  - 단편적인 파일 제출에 치중하지 않고, 이번 미션에서 구축한 스트레스 시뮬레이션 결과와 성능 평가 지표(대시보드 로딩 시간 2초 이내, 경보 발령 정확도 95% 이상 달성 여부 등)를 객관적으로 검수하여 후속 미션(미션 9: AAR 기반 결핍 경보 지연 원인 분석 및 경보 임계 파라미터 재보정)으로 온전히 인계할 수 있도록 패키지를 정비합니다.

#### 2. 세부 수행 절차

- **핵심 산출 파일 패키지 확정 및 제출:**
  - 대시보드 소스 코드인 src/dashboard/str01_app.py , 스트레스 시뮬레이션 분석 보고서인 reports/str01_stress_report.md , 결과 데이터 파일인 outputs/M08/res.csv , 그리고 실습 기록 서식인 forms/STR01_M08_forms.md 의 실제 작성 파일들을 최종 점검하고 제출 상태로 정리합니다.
- **코드 실행 결과 및 AI 대화 기록 연계 보존:**
  - evaluate 함수 구현 및 스트레스 시뮬레이션 결과, 그리고 분석 과정에서 AI 에이전트와 나눈 핵심 질의응답 및 코드 선택 근거 대화 내용을 실습 대화 보존 위치에 명확히 연결합니다.
- **상태별 최종 판단 기록 작성:**
  - 정량적 목표 달성 여부, 현장 여건상 미처 측정하지 못한 영역, 문서 작성 완료 상태, 그리고 실제 교관 승인 대기 상태를 투명하게 구분하여 서식( forms/ST…

#### 완료 확인:

- 이 활동의 확인 근거 M08 완료: src/dashboard/str01_app.py, reports/str01_stress_report.md, outputs/M08/res.csv, forms/STR01_M08_forms.md의실제작성·최종값·대화보존위치·선택이유가연결되는가?
- 미달·미측정·문서미작성·실제승인대기를정확히표시했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

#### 내가 기록할 것:

- 직접 확인한 입력·원본 조건:
- AI에 실제로 요청한 질문과 받은 답변의 위치:
- 실제 실행 결과/오류 메시지:
- 잘못된 설명, 수정한 코드, 수정 이유:


## 🔍 4. 평가 기준과 확인할 자료

- [정량적 평가] [선택] 충격 시나리오·로딩 시간 — 대시보드 로딩 시간 2초 이내, 경보 발령 정확도 95% 이상이며, 의사결정 지원에 맞게 경보 위계가 시각적으로 분명한가. PASS 증거 제공된조건·단위·기간·분모와실제수치/건수/항목근거를연결합니다. 목표미달·0기준·미측정은그대로표시합니다. 보완 방법 기준별빠진항목과미달원인을찾아수정하고재검증합니다.
- **[정성적 평가] AI 활용: 본인 계획·실제 대화·코드 변경·검토3건·보완·재실행·전후 비교·최종 선택.**
  - **PASS 증거**
    - **수행 조건과 판정 기준**
      - 시험은 추상적인 소모 배율과 보급 50% 감소입니다.
      - 정확도 분모에는 관찰 재고 0건을 포함하며 이들은 모델 예측 성공이 아니라 규칙 처리입니다.
      - 경보 여부 일치율 95%와 4등급 일치율은 다릅니다.
      - 50회 갱신 시간은 피처 생성·예측·집계·문자열 생성의 로컬 시간이며 웹 서비스 전체 로딩 시간이 아닙니다.
      - 기간을 늘리면 make_samples의 hi도 조정하고 종료 범위를 기록합니다.
    - **보완 방법**
      - 보완 방법 근거없는원인·규정·완료추정은수정하거나확인필요로보류합니다.
- **[진행 추적 및 완료 기준] 정성·검증:**
  - **수행 조건과 판정 기준**
    - 시험은 추상적인 소모 배율과 보급 50% 감소입니다.
    - 정확도 분모에는 관찰 재고 0건을 포함하며 이들은 모델 예측 성공이 아니라 규칙 처리입니다.
    - 경보 여부 일치율 95%와 4등급 일치율은 다릅니다.
    - 50회 갱신 시간은 피처 생성·예측·집계·문자열 생성의 로컬 시간이며 웹 서비스 전체 로딩 시간이 아닙니다.
    - 기간을 늘리면 make_samples의 hi도 조정하고 종료 범위를 기록합니다.
  - **제출 파일 및 완료 기록**
    - 파일: src/dashboard/str01_app.py, reports/str01_stress_report.md, outputs/M08/res.csv, forms/STR01_M08_forms.md 실제AI요청·답변·근거대조3건이상·보완·재확인·본인최종선택과작성/실행기록이연결됩니다.
    - 최종ZIP생성은실제학습완료·교관승인 인증이아닙니다.
  - **보완 방법**
    - 보완 방법 미작성·미실행·검토누락을보완하고교관검토상태를별도로표시합니다.
**실제 출력 없이 달성으로 표시하지 않습니다.**

In [ ]:
# @title ⑤ 개별 산출물·학습기록 통합 입력, 저장 및 ZIP 다운로드 (실행 버튼을 누르세요) { display-mode: "form" }
# 입력 폼이 나타나면 소스코드 대신 버튼을 이용하세요. Colab/Jupyter에서 ipywidgets 필요.
from pathlib import Path

if not (globals().get('STR01_PROJECT_READY', False) and globals().get('STR01_PROJECT_DIR') and Path(STR01_PROJECT_DIR).is_dir()):
    print('먼저 ① 실습자료 자동 준비 및 환경 설정 셀을 실행한 뒤 이 셀을 다시 실행하세요. ZIP 업로드는 필요 없습니다.')
else:
    def _load_notebook_widgets():
        from importlib import import_module, invalidate_caches
        import subprocess
        import sys

        try:
            return import_module('ipywidgets')
        except ImportError:
            print('이 노트북 커널에 ipywidgets가 없어 설치합니다.')

        try:
            subprocess.run(
                [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
                 '--retries', '0', '--timeout', '10', 'ipywidgets'],
                check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=60,
            )
            invalidate_caches()
            return import_module('ipywidgets')
        except (subprocess.SubprocessError, OSError, ImportError):
            print('이 커널에서 ipywidgets를 설치하거나 불러오지 못했습니다.')
            print('설치 가능한 환경에서는 새 셀에서 %pip install ipywidgets 를 실행한 뒤 이 셀을 다시 실행하세요.')
            print('설치할 수 없는 환경에서는 제공된 Markdown 작성 양식을 사용하세요.')
            return None

    widgets = _load_notebook_widgets()
    if widgets is not None:
        import json, csv, io, ast, zipfile, datetime, pathlib
        from pathlib import Path
        import os
        from urllib.parse import quote
        from IPython.display import HTML, display, Markdown, clear_output

        MISSION_CODE = 8
        DELIVERABLE_PATHS = ['src/dashboard/str01_app.py', 'reports/str01_stress_report.md', 'outputs/M08/res.csv', 'forms/STR01_M08_forms.md']
        PROJECT_NAME = 'STR01'
        WORK_ROOT = Path(STR01_PROJECT_DIR).resolve()

        RECORD_DIR = WORK_ROOT / f'STR01_M{MISSION_CODE:02d}_학습기록'
        RECORD_DIR.mkdir(parents=True, exist_ok=True)
        ZIP_FILENAME = f'STR01_m{MISSION_CODE:02d}_review.zip'

        # 반드시 CMS 지정 경로만 사용하며 상대 경로의 작업공간 이탈을 방지합니다.
        def safe_path(relative):
            target = (WORK_ROOT / relative).resolve()
            if not target.is_relative_to(WORK_ROOT):
                raise ValueError('작업공간 밖의 경로는 사용할 수 없습니다.')
            return target

        def valid_text(path, content):
            suffix = Path(path).suffix.lower()
            if suffix == '.json':
                json.loads(content)
            elif suffix == '.ipynb':
                doc = json.loads(content)
                if not isinstance(doc, dict) or not isinstance(doc.get('cells'), list) or 'nbformat' not in doc:
                    raise ValueError('노트북 JSON에 cells 또는 nbformat 정보가 없습니다.')
            elif suffix == '.py':
                ast.parse(content)
            elif suffix == '.csv':
                if not content.strip():
                    raise ValueError('CSV 데이터가 비어 있습니다.')
                rows = list(csv.reader(io.StringIO(content)))
                if not rows or not rows[0] or not any(x.strip() for x in rows[0]):
                    raise ValueError('CSV 첫 행에 열 이름을 입력하세요.')
                if any(len(row) != len(rows[0]) for row in rows if row):
                    raise ValueError('CSV 열 개수가 행마다 다릅니다.')

        # 산출물 입력 화면 — 파일별 별도 탭
        editors = {}
        file_tabs = []
        file_buttons = {}
        for relative in DELIVERABLE_PATHS:
            suffix = Path(relative).suffix.lower()
            hint = {
                '.md': '보고서·양식의 제목, 분석 결과, 근거와 수정 이유를 Markdown으로 작성합니다.',
                '.csv': '첫 줄에 열 이름, 다음 줄부터 데이터 행을 쉼표로 구분하여 입력합니다.',
                '.json': '키와 값이 있는 유효한 JSON 형식으로 실제 결과를 입력합니다.',
                '.py': '직접 구현하고 확인한 Python 소스코드를 입력합니다.',
                '.ipynb': '노트북 JSON 원문을 입력하거나 기존 파일에서 불러옵니다. JSON 노트북 형식을 유지하세요.'
            }.get(suffix, '실제 작성한 내용을 입력합니다.')
            editor = widgets.Textarea(value='', placeholder='여기에 내용을 작성하세요. 입력하지 않은 파일은 저장하지 않습니다.',
                                      layout=widgets.Layout(width='100%', height='250px'))
            status = widgets.HTML('<span style="color:#777">아직 저장하지 않았습니다.</span>')
            load_btn = widgets.Button(description='기존 파일 불러오기', icon='folder-open', button_style='')
            save_btn = widgets.Button(description='이 파일 저장', icon='save', button_style='success')
            editors[relative] = {'editor': editor, 'status': status, 'loaded': False, 'saved': False, 'dirty': False, 'saved_value': ''}
            def on_edit(change, key=relative):
                state = editors[key]
                if state['loaded']:
                    state['dirty'] = True
                    state['status'].value = '<span style="color:#a65">● 수정됨 — 저장 버튼을 누르세요.</span>'
            editor.observe(on_edit, names='value')
            def load_clicked(button, key=relative):
                state = editors[key]
                target = safe_path(key)
                if not target.is_file():
                    state['status'].value = '<span style="color:#a22">파일이 아직 없습니다. 직접 입력 후 저장하세요.</span>'
                    return
                if state['dirty']:
                    state['status'].value = '<span style="color:#a22">미저장 내용이 있습니다. 불러오기 전에 저장하세요.</span>'
                    return
                try:
                    state['loaded'] = False
                    state['saved_value'] = target.read_text(encoding='utf-8')
                    state['editor'].value = state['saved_value']
                    state['loaded'] = True
                    state['dirty'] = False
                    state['saved'] = True
                    state['status'].value = '<span style="color:#286">파일을 불러왔습니다. 변경하면 다시 저장하세요.</span>'
                except Exception as exc:
                    state['status'].value = '<span style="color:#a22">불러오기 오류: '+str(exc)+'</span>'
            def save_clicked(button, key=relative):
                save_one(key)
            load_btn.on_click(load_clicked)
            save_btn.on_click(save_clicked)
            file_buttons[relative] = {'load':load_btn,'save':save_btn}
            file_tabs.append(widgets.VBox([
                widgets.HTML('<b>저장 경로:</b> <code>'+relative+'</code><br><span>'+hint+'</span><br><span style="color:#a55">기존 파일이 있다면 저장 시 덮어씁니다. 실습 완료·승인과는 다릅니다.</span>'),
                editor, widgets.HBox([load_btn, save_btn]), status
            ]))

        def save_one(relative):
            state = editors[relative]
            value = state['editor'].value
            if not value.strip():
                state['status'].value = '<span style="color:#a22">비어 있어 저장하지 않았습니다.</span>'
                return False
            try:
                valid_text(relative, value)
                target = safe_path(relative)
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_text(value, encoding='utf-8')
                state['saved_value'] = value
                state['loaded'] = True
                state['dirty'] = False
                state['saved'] = True
                state['status'].value = '<span style="color:green">✓ 저장 완료: '+relative+'</span>'
                return True
            except Exception as exc:
                state['status'].value = '<span style="color:#a22">저장 실패: '+str(exc)+'</span>'
                return False

        # 학습기록 및 AI 원문, 자료 대조, 검토 3건 이상
        record_specs = [
         ('plan','학습 계획·수행 기록','이번 미션에서 직접 수행한 내용, 실행 조건 및 최초 결과를 작성하세요.'),
         ('ai_request','실제 AI 질문 원문','실제로 입력한 질문 원문만 작성하세요. 사용하지 않았다면 비워 둡니다.'),
         ('ai_response','실제 받은 AI 답변 원문','실제로 받은 답변 전체를 붙여 넣으세요. 임의로 생성한 대화는 기록하지 않습니다.'),
         ('evidence','실제 출력·자료 대조 근거','실제 출력값, 원자료 위치, 전후 비교 및 계산 근거를 적으세요.'),
         ('revision','검토 및 수정 내용 종합','검토 3건의 발견사항과 코드를 수정·재실행한 기록을 종합하세요.'),
         ('decision','최종 판단과 미달·미측정·미실시 사유','달성 여부와 남은 부분, 근거 및 추가 확인사항을 적으세요.'),
        ]
        record_widgets = {}
        record_items = []
        for key, title, hint in record_specs:
            field = widgets.Textarea(placeholder=hint, layout=widgets.Layout(width='100%',height='115px'))
            record_widgets[key] = field
            record_items.append(widgets.VBox([widgets.HTML('<b>'+title+'</b><br><small>'+hint+'</small>'),field]))

        reviews = []
        for i in range(3):
            claim = widgets.Textarea(placeholder=f'{i+1}번째 AI 주장·코드 선택 또는 점검 대상',layout=widgets.Layout(width='100%',height='65px'))
            ground = widgets.Textarea(placeholder='원자료·실행 출력의 위치 및 비교한 근거',layout=widgets.Layout(width='100%',height='65px'))
            action = widgets.Textarea(placeholder='맞음·수정·보류, 수정 내용 및 재확인 결과',layout=widgets.Layout(width='100%',height='65px'))
            reviews.append((claim,ground,action))
            record_items.append(widgets.VBox([widgets.HTML(f'<b>검토 {i+1} (최소 3건 중 {i+1}번째)</b>'),claim,ground,action]))

        status_check = widgets.SelectMultiple(options=['달성','미달','미측정','미실시','추가 확인 필요'],
                                              value=(),description='수행 상태',layout=widgets.Layout(width='70%',height='115px'))
        approval = widgets.Dropdown(options=['미확인 / 교관 승인 대기','실제 교관 검토 완료 (기록 확인 필요)'],
                                    value='미확인 / 교관 승인 대기', description='교관 상태',layout=widgets.Layout(width='85%'))
        record_notice = widgets.HTML('<span style="color:#666">입력 후 [학습기록 저장]을 누르세요. 미작성·미측정은 그대로 표시합니다.</span>')
        record_dirty = {'value': False}
        def _mark_record_dirty(change):
            record_dirty['value'] = True
            record_notice.value = '<span style="color:#a65">● 학습기록에 저장하지 않은 변경이 있습니다.</span>'
        for field in list(record_widgets.values()) + [v for group in reviews for v in group] + [status_check,approval]:
            field.observe(_mark_record_dirty,names='value')

        # 앞서 작성한 학습기록을 자동으로 불러와 이어서 수정할 수 있습니다.
        previous_audit = RECORD_DIR/'learner_audit.json'
        if previous_audit.is_file():
            try:
                previous = json.loads(previous_audit.read_text(encoding='utf-8'))
                for key, field in record_widgets.items():
                    field.value = str(previous.get(key,'') or '')
                for group, past in zip(reviews, previous.get('reviews',[])):
                    for widget, key in zip(group,('claim','evidence','revision_and_recheck')):
                        widget.value = str(past.get(key,'') or '')
                status_check.value = tuple(x for x in previous.get('status',[]) if x in status_check.options)
                if previous.get('instructor_approval') in approval.options:
                    approval.value = previous['instructor_approval']
                record_dirty['value'] = False
                record_notice.value = '<span style="color:#286">이전 학습기록을 불러왔습니다. 수정 후 다시 저장하세요.</span>'
            except Exception as exc:
                record_notice.value = '<span style="color:#a22">기존 학습기록 읽기 오류: '+str(exc)+'</span>'

        def audit_object():
            return {
                'project': PROJECT_NAME, 'mission': MISSION_CODE,
                'plan':record_widgets['plan'].value, 'ai_request':record_widgets['ai_request'].value,
                'ai_response':record_widgets['ai_response'].value, 'evidence':record_widgets['evidence'].value,
                'revision':record_widgets['revision'].value, 'decision':record_widgets['decision'].value,
                'reviews':[{'claim':a.value,'evidence':b.value,'revision_and_recheck':c.value} for a,b,c in reviews],
                'status':list(status_check.value),'instructor_approval':approval.value,
                'saved_at':datetime.datetime.now().isoformat(timespec='seconds')
            }

        def save_audit(button=None):
            try:
                record = audit_object()
                (RECORD_DIR/'learner_audit.json').write_text(json.dumps(record,ensure_ascii=False,indent=2),encoding='utf-8')
                (RECORD_DIR/'AI_대화원문.txt').write_text('실제 AI 질문 원문\n'+record['ai_request']+'\n\n실제 AI 답변 원문\n'+record['ai_response'],encoding='utf-8')
                sections = [f'# STR01 미션 {MISSION_CODE:02d} 학습기록']
                for name,key in [('학습 계획·수행 기록','plan'),('실제 출력·자료 대조 근거','evidence'),('검토 및 수정 내용','revision'),('최종 판단','decision')]:
                    sections.append('## '+name+'\n'+(record[key] or '(미작성)'))
                for i,item in enumerate(record['reviews'],1):
                    sections.append(f"## 검토 {i}\n- 검토 대상: {item['claim'] or '(미작성)'}\n- 대조 근거: {item['evidence'] or '(미작성)'}\n- 수정 및 재확인: {item['revision_and_recheck'] or '(미작성)'}")
                sections.append('## 상태\n'+(', '.join(record['status']) or '미선택')+'\n\n교관 상태: '+record['instructor_approval'])
                (RECORD_DIR/'학습기록.md').write_text('\n\n'.join(sections)+'\n',encoding='utf-8')
                record_dirty['value'] = False
                completed = sum(all(v.strip() for v in [a.value,b.value,c.value]) for a,b,c in reviews)
                record_notice.value = f'<span style="color:green">✓ 학습기록 저장 완료. 근거가 모두 입력된 검토: {completed}/3건. 저장은 PASS나 교관 승인과 다릅니다.</span>'
                return True
            except Exception as exc:
                record_notice.value = '<span style="color:#a22">학습기록 저장 실패: '+str(exc)+'</span>'
                return False

        save_audit_btn = widgets.Button(description='학습기록 저장',icon='save',button_style='success')
        save_audit_btn.on_click(save_audit)
        record_tab = widgets.VBox(record_items+[widgets.HTML('<b>최종 수행 상태</b> — 해당되는 항목을 선택하세요. 선택만으로 자동 평가되지 않습니다.'),status_check,approval,save_audit_btn,record_notice])

        tabs = widgets.Tab(children=file_tabs+[record_tab])
        for i, path in enumerate(DELIVERABLE_PATHS):
            tabs.set_title(i, f'산출물 {i+1}')
        tabs.set_title(len(file_tabs),'학습기록')

        summary_output = widgets.Output()
        all_btn = widgets.Button(description='작성한 산출물 전체 저장', icon='save',button_style='info')
        zip_btn = widgets.Button(description='저장 파일 ZIP 생성',icon='archive',button_style='warning')
        dl_btn = widgets.Button(description='ZIP 다운로드',icon='download',button_style='')

        def save_all(button):
            ok = 0
            for rel in DELIVERABLE_PATHS:
                if editors[rel]['editor'].value.strip() and save_one(rel):
                    ok += 1
            with summary_output:
                summary_output.clear_output()
                print(f'산출물 {ok}개 저장 완료. 파일별 상태를 확인하세요. 학습기록은 별도 [학습기록 저장] 버튼을 누르세요.')

        def build_zip(button):
            with summary_output:
                summary_output.clear_output()
                if record_dirty['value']:
                    print('❌ 학습기록에 미저장 변경 내용이 있습니다. [학습기록 저장]을 먼저 누르세요.')
                    return
                dirty = [p for p,s in editors.items() if s['dirty']]
                if dirty:
                    print('❌ 미저장 변경 내용이 있습니다. 아래 파일들을 먼저 저장하세요:')
                    for p in dirty: print(' -',p)
                    return
                # 기존에 제공된 파일은 ZIP에 임의로 포함하지 않고 현재 화면에서 불러오거나 저장한 파일만 수집합니다.
                saved = [p for p,s in editors.items() if s['saved'] and safe_path(p).is_file()]
                audit_files = [p for p in RECORD_DIR.rglob('*') if p.is_file()]
                if not saved and not audit_files:
                    print('❌ 저장된 결과물이 없습니다. 산출물 또는 학습기록을 저장하세요.')
                    return
                dest = WORK_ROOT / ZIP_FILENAME
                with zipfile.ZipFile(dest,'w',compression=zipfile.ZIP_DEFLATED) as archive:
                    for rel in saved: archive.write(safe_path(rel), arcname=rel)
                    for f in audit_files: archive.write(f, arcname=str(f.relative_to(WORK_ROOT)))
                print('✅ ZIP 생성:', dest)
                print('산출물:',len(saved),'개 / 학습기록:',len(audit_files),'개')
                print('※ ZIP 생성은 학습 완료 또는 교관 승인을 의미하지 않습니다.')

        def download_zip(button):
            with summary_output:
                summary_output.clear_output()
                target = WORK_ROOT / ZIP_FILENAME
                if not target.is_file():
                    print('먼저 [저장 파일 ZIP 생성]을 눌러 주세요.')
                    return
                try:
                    from google.colab import files as colab_files
                    colab_files.download(str(target))
                    print('다운로드를 요청했습니다.')
                except ImportError:
                    start_dir = globals().get('notebook_directory', WORK_ROOT.parent)
                    href = quote(Path(os.path.relpath(target, start_dir)).as_posix())
                    display(HTML(f'<a href="{href}" download>ZIP 다운로드: {target.name}</a>'))

        all_btn.on_click(save_all)
        zip_btn.on_click(build_zip)
        dl_btn.on_click(download_zip)

        print(f'✅ STR01 미션 {MISSION_CODE:02d} 산출물·학습기록 입력 화면')
        display(Markdown('**사용 순서:** ① 각 산출물 탭에서 작성하거나 기존 파일 불러오기 → ② 파일별 저장 → ③ 학습기록 탭에서 실제 AI 원문·근거·검토 3건·최종 상태 입력 및 저장 → ④ ZIP 생성 → ⑤ 다운로드.'))
        display(Markdown('**주의:** 원본 실습의 TODO는 직접 실행해야 합니다. 입력창은 실제 수행 내용을 기록하기 위한 기능이며, 미작성 결과를 자동으로 완성하지 않습니다.'))
        display(tabs,widgets.HBox([all_btn,zip_btn,dl_btn]),summary_output)
